In [21]:
# IMPORTS
from pathlib import Path
import os
import zipfile
import cdsapi
import numpy as np
import pandas as pd
import xarray as xr
import scipy 

In [17]:
# RUTAS DE TRABAJO
BASE = Path(r"C:\Users\Gambo\Documents\Master Aaron\TFM UCM")

DATA = BASE / "Data"
FIGURES = BASE / "Figures"

ruta_firespain = DATA / "firespain025_WGS84.nc"
ERA5_RAW = DATA / "era5"
ERA5_EXTRACTED = DATA / "era5_extracted"

# DESCARGA DE ERA5-LAND

In [ ]:
# DESCARGA DE ERA5-LAND (EJECUTADA UNA ÚNICA VEZ)
# Los archivos se descargaron mediante la API de Copernicus CDS.
# Este bloque se conserva únicamente por reproducibilidad si se desea volver a descargar los datos
# cambiar Descargar_ERA5 a True y ejecutar el bloque.
# =============================================================================
DESCARGAR_ERA5 = False

if DESCARGAR_ERA5:
    c = cdsapi.Client()
    for year in range(2008, 2023):
        print(f"Descargando {year}...")
        meses = (["01"] if year == 2022 else [f"{m:02d}" for m in range(1, 13)])

        c.retrieve(
            "reanalysis-era5-land",
            {
                "variable": [
                    "volumetric_soil_water_layer_1",
                    "volumetric_soil_water_layer_2",
                    "2m_temperature",
                    "total_precipitation",
                ],
                "year": str(year),
                "month": meses,
                "day": [f"{d:02d}" for d in range(1, 32)],
                "time": "12:00",
                "data_format": "netcdf",
                "area": [44.0, -9.5, 35.0, 4.5],
            },
            ERA5_RAW / f"era5_land_{year}.nc",
        )

EXTRACCIÓN DE LOS ARCHIVOS

In [4]:

for year in range(2008, 2023):
    zip_path = ERA5_RAW / f"era5_land_{year}.nc"
    out_path = ERA5_EXTRACTED / f"era5_land_{year}.nc"
    with zipfile.ZipFile(zip_path) as z:
        with z.open("data_0.nc") as src, open(out_path, "wb") as dst:
            dst.write(src.read())
        print(f"{year} -> {out_path}")

print("\nExtracción completada.")

2008 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2008.nc
2009 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2009.nc
2010 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2010.nc
2011 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2011.nc
2012 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2012.nc
2013 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2013.nc
2014 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2014.nc
2015 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2015.nc
2016 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2016.nc
2017 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_land_2017.nc
2018 -> C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5_extracted\era5_l

VERIFICACIÓN PREVIA

In [5]:
archivos = sorted(ERA5_EXTRACTED.glob("era5_land_*.nc"))

for f in archivos:
    ds = xr.open_dataset(f)
    print(
        f"{f.name} | "
        f"tiempo: {len(ds.valid_time)} días | "
        f"lat: {len(ds.latitude)} | "
        f"lon: {len(ds.longitude)} | "
        f"vars: {list(ds.data_vars)}"
    )

    ds.close()

era5_land_2008.nc | tiempo: 366 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2009.nc | tiempo: 365 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2010.nc | tiempo: 365 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2011.nc | tiempo: 365 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2012.nc | tiempo: 366 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2013.nc | tiempo: 365 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2014.nc | tiempo: 365 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2015.nc | tiempo: 365 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2016.nc | tiempo: 366 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2017.nc | tiempo: 365 días | lat: 91 | lon: 141 | vars: ['swvl1', 'swvl2', 't2m', 'tp']
era5_land_2018.nc | 

CONCATENACIÓN

In [6]:
# Archivos  previamente extraídos
archivos = sorted(ERA5_EXTRACTED.glob("era5_land_*.nc"))

# Apertura y concatenación por coordenadas
era5 = xr.open_mfdataset(archivos, combine="by_coords", parallel=False)

# Renombrar la dimensión temporal para mantener consistencia con el resto de datasets del proyecto
era5 = era5.rename({"valid_time": "time"})

# Eliminar coordenadas auxiliares innecesarias
era5 = era5.drop_vars(["expver", "number"], errors="ignore")

print("DATASET CONCATENADO")
print(era5)

print(f"\nPeriodo: {era5.time.min().values} → {era5.time.max().values}")
print(f"Número de días: {len(era5.time)}")

DATASET CONCATENADO
<xarray.Dataset> Size: 1GB
Dimensions:    (time: 5145, latitude: 91, longitude: 141)
Coordinates:
  * time       (time) datetime64[ns] 41kB 2008-01-01T12:00:00 ... 2022-01-31T...
  * latitude   (latitude) float64 728B 44.0 43.9 43.8 43.7 ... 35.2 35.1 35.0
  * longitude  (longitude) float64 1kB -9.5 -9.4 -9.3 -9.2 ... 4.2 4.3 4.4 4.5
Data variables:
    swvl1      (time, latitude, longitude) float32 264MB dask.array<chunksize=(183, 46, 71), meta=np.ndarray>
    swvl2      (time, latitude, longitude) float32 264MB dask.array<chunksize=(183, 46, 71), meta=np.ndarray>
    t2m        (time, latitude, longitude) float32 264MB dask.array<chunksize=(183, 46, 71), meta=np.ndarray>
    tp         (time, latitude, longitude) float32 264MB dask.array<chunksize=(183, 46, 71), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF

VERIFICACIÓN DE LA SERIE TEMPORAL

In [7]:
# Comprobar cobertura del año 2022
dias_2022 = era5.sel(time=era5.time.dt.year == 2022)
print(f"Días disponibles en 2022: {len(dias_2022.time)}")
print(f"Periodo: "f"desde: {dias_2022.time.values[0]} hasta:  "f"{dias_2022.time.values[-1]}")

Días disponibles en 2022: 31
Periodo: desde: 2022-01-01T12:00:00.000000000 hasta:  2022-01-31T12:00:00.000000000


In [ ]:
# Mantener únicamente el período de estudio
# Para coincidir con el dataset FIRESPAIN-025
era5 = era5.sel(time=slice("2008-01-01","2022-01-01T12:00:00"))
print(f"\nNúmero de días tras el recorte: {len(era5.time)}")


Número de días tras el recorte: 5115


In [9]:
# Eliminar la componente horaria (12:00 → 00:00)
era5["time"] = era5.time.dt.floor("D")

print("\nPrimeras fechas:")
print(era5.time.values[:3])

print("\nÚltimas fechas:")
print(era5.time.values[-3:])


Primeras fechas:
['2008-01-01T00:00:00.000000000' '2008-01-02T00:00:00.000000000'
 '2008-01-03T00:00:00.000000000']

Últimas fechas:
['2021-12-30T00:00:00.000000000' '2021-12-31T00:00:00.000000000'
 '2022-01-01T00:00:00.000000000']


In [11]:
# Verificar continuidad temporal
tiempo = pd.DatetimeIndex(era5.time.values)
diferencias = np.diff(tiempo)
gaps = tiempo[1:][diferencias != pd.Timedelta(days=1)]

if len(gaps) == 0:
    print("Serie temporal continua (sin días faltantes).")
else:
    print(f"Se detectaron {len(gaps)} discontinuidades:")
    print(gaps)

Serie temporal continua (sin días faltantes).


DERIVACIÓN SWI DESDE swvl1 Y swvl2

In [12]:
def calc_swi(swvl: np.ndarray, T: int) -> np.ndarray:
    """
    Filtro exponencial recursivo sobre serie temporal de humedad del suelo.
    swvl: array (time, lat, lon)
    T:    factor de tiempo en días
    """
    swi = np.full_like(swvl, np.nan)
    gain = np.full(swvl.shape[1:], np.nan)  # (lat, lon)

    # Inicializar con primer valor válido
    swi[0] = swvl[0]
    gain[:] = 1.0

    decay = np.exp(-1.0 / T)

    for t in range(1, len(swvl)):
        gain = gain / (gain + decay)
        swi[t] = swi[t-1] + gain * (swvl[t] - swi[t-1])

    return swi

# Cargar swvl en memoria 
swvl1 = era5["swvl1"].values  # (5115, 91, 141)
swvl2 = era5["swvl2"].values

# Calcular SWI para T=10, T=20, T=40
print("Derivando SWI:")
for T in [10, 20, 40]:
    for capa, swvl in [("swvl1", swvl1), ("swvl2", swvl2)]:
        nombre = f"swi_T{T}_{capa}"
        print(f"- {nombre}")
        era5[nombre] = xr.DataArray(
            calc_swi(swvl, T),
            dims=["time", "latitude", "longitude"],
            coords={
                "time": era5.time,
                "latitude": era5.latitude,
                "longitude": era5.longitude
            },
            attrs={"long_name": f"SWI T={T} desde {capa}", "units": "m3/m3"}
        )

print("\nSWI derivado")
print(era5)

Derivando SWI:
- swi_T10_swvl1
- swi_T10_swvl2
- swi_T20_swvl1
- swi_T20_swvl2
- swi_T40_swvl1
- swi_T40_swvl2

SWI derivado
<xarray.Dataset> Size: 3GB
Dimensions:        (time: 5115, latitude: 91, longitude: 141)
Coordinates:
  * time           (time) datetime64[ns] 41kB 2008-01-01 ... 2022-01-01
  * latitude       (latitude) float64 728B 44.0 43.9 43.8 ... 35.2 35.1 35.0
  * longitude      (longitude) float64 1kB -9.5 -9.4 -9.3 -9.2 ... 4.3 4.4 4.5
Data variables:
    swvl1          (time, latitude, longitude) float32 263MB dask.array<chunksize=(183, 46, 71), meta=np.ndarray>
    swvl2          (time, latitude, longitude) float32 263MB dask.array<chunksize=(183, 46, 71), meta=np.ndarray>
    t2m            (time, latitude, longitude) float32 263MB dask.array<chunksize=(183, 46, 71), meta=np.ndarray>
    tp             (time, latitude, longitude) float32 263MB dask.array<chunksize=(183, 46, 71), meta=np.ndarray>
    swi_T10_swvl1  (time, latitude, longitude) float32 263MB nan nan ... 

In [13]:
# Verificar NaN iniciales en SWI
for T in [10, 20, 40]:
    var = f"swi_T{T}_swvl1"
    # Contar días donde hay algún NaN espacial
    nans_por_dia = np.isnan(era5[var].values).any(axis=(1, 2))
    dias_con_nan = nans_por_dia.sum()
    print(f"swi_T{T}: {dias_con_nan} días con algún NaN")

swi_T10: 5115 días con algún NaN
swi_T20: 5115 días con algún NaN
swi_T40: 5115 días con algún NaN


In [14]:
# ¿Los NaN están siempre en las mismas celdas espaciales?
nan_mask_swi = np.isnan(era5["swi_T10_swvl1"].values)

# Cuántos días tiene NaN cada celda
nan_por_celda = nan_mask_swi.sum(axis=0)

print(f"Celdas con NaN en TODOS los días: {(nan_por_celda == 5115).sum()}")
print(f"Celdas con NaN en ALGÚN día:      {(nan_por_celda > 0).sum()}")
print(f"Celdas con CERO NaN:              {(nan_por_celda == 0).sum()}")

# ¿Los NaN en swvl1 original coinciden?
nan_mask_swvl = np.isnan(era5["swvl1"].values)
nan_por_celda_swvl = nan_mask_swvl.sum(axis=0)
print(f"\nCeldas con NaN en swvl1 original: {(nan_por_celda_swvl > 0).sum()}")

Celdas con NaN en TODOS los días: 4870
Celdas con NaN en ALGÚN día:      4870
Celdas con CERO NaN:              7961

Celdas con NaN en swvl1 original: 4870


AGREGACIÓN ESPACIAL ERA5

In [22]:
# Grid objetivo — extraído directamente de FIRESPAIN para garantizar alineación exacta
fr25 = xr.open_dataset(ruta_firespain)

lat_target = fr25.latitude.values   # 34 valores
lon_target = fr25.longitude.values  # 54 valores

print(f"Grid FIRESPAIN: {len(lat_target)} lat × {len(lon_target)} lon")
print(f"Lat: {lat_target[0]:.3f} → {lat_target[-1]:.3f}")
print(f"Lon: {lon_target[0]:.3f} → {lon_target[-1]:.3f}")

# Agregar por interpolación conservativa — media de celdas 0.1° dentro de cada celda 0.25°
era5_025 = era5.interp(
    latitude=lat_target,
    longitude=lon_target,
    method="linear"
)

print(f"\nERA5 agregado a 0.25°")
print(era5_025)

Grid FIRESPAIN: 34 lat × 54 lon
Lat: 43.601 → 35.351
Lon: -9.135 → 4.115

ERA5 agregado a 0.25°
<xarray.Dataset> Size: 601MB
Dimensions:        (time: 5115, latitude: 34, longitude: 54)
Coordinates:
  * time           (time) datetime64[ns] 41kB 2008-01-01 ... 2022-01-01
  * latitude       (latitude) float64 272B 43.6 43.35 43.1 ... 35.85 35.6 35.35
  * longitude      (longitude) float64 432B -9.135 -8.885 -8.635 ... 3.865 4.115
Data variables:
    swvl1          (time, latitude, longitude) float32 38MB dask.array<chunksize=(46, 17, 54), meta=np.ndarray>
    swvl2          (time, latitude, longitude) float32 38MB dask.array<chunksize=(46, 17, 54), meta=np.ndarray>
    t2m            (time, latitude, longitude) float32 38MB dask.array<chunksize=(46, 17, 54), meta=np.ndarray>
    tp             (time, latitude, longitude) float32 38MB dask.array<chunksize=(46, 17, 54), meta=np.ndarray>
    swi_T10_swvl1  (time, latitude, longitude) float64 75MB nan nan ... 0.06686
    swi_T10_swvl2  (time

In [23]:
# Reconvertir SWI a float32
for var in era5_025.data_vars:
    if era5_025[var].dtype == np.float64:
        era5_025[var] = era5_025[var].astype(np.float32)

# Verificar tipos
for var in era5_025.data_vars:
    print(f"{var}: {era5_025[var].dtype}")

swvl1: float32
swvl2: float32
t2m: float32
tp: float32
swi_T10_swvl1: float32
swi_T10_swvl2: float32
swi_T20_swvl1: float32
swi_T20_swvl2: float32
swi_T40_swvl1: float32
swi_T40_swvl2: float32


EXPORTAR DATASET

In [ ]:
ruta_era5_clean = ERA5_RAW / "era5_land_clean.nc"
era5_025.to_netcdf(ruta_era5_clean)

print("DATASET GUARDADO")
print(f"Ruta: {ruta_era5_clean}")
print(f"Tamaño: {ruta_era5_clean.stat().st_size / 1e6:.1f} MB")

DATASET GUARDADO
Ruta: C:\Users\Gambo\Documents\Master Aaron\TFM UCM\Data\era5\era5_land_clean.nc
Tamaño: 375.7 MB


VERIFICACIÓN DEL ARCHIVO GENERADO

In [ ]:
era5_check = xr.open_dataset(ruta_era5_clean)

print("\nResumen del archivo generado:")
print(era5_check)

print(f"\nPeriodo: {era5_check.time.min().values} → {era5_check.time.max().values}")
print(f"Variables: {list(era5_check.data_vars)}")

era5_check.close()


Resumen del archivo generado:
<xarray.Dataset> Size: 376MB
Dimensions:        (time: 5115, latitude: 34, longitude: 54)
Coordinates:
  * time           (time) datetime64[ns] 41kB 2008-01-01 ... 2022-01-01
  * latitude       (latitude) float64 272B 43.6 43.35 43.1 ... 35.85 35.6 35.35
  * longitude      (longitude) float64 432B -9.135 -8.885 -8.635 ... 3.865 4.115
Data variables:
    swvl1          (time, latitude, longitude) float32 38MB ...
    swvl2          (time, latitude, longitude) float32 38MB ...
    t2m            (time, latitude, longitude) float32 38MB ...
    tp             (time, latitude, longitude) float32 38MB ...
    swi_T10_swvl1  (time, latitude, longitude) float32 38MB ...
    swi_T10_swvl2  (time, latitude, longitude) float32 38MB ...
    swi_T20_swvl1  (time, latitude, longitude) float32 38MB ...
    swi_T20_swvl2  (time, latitude, longitude) float32 38MB ...
    swi_T40_swvl1  (time, latitude, longitude) float32 38MB ...
    swi_T40_swvl2  (time, latitude, longi

# DESCARGA MDT